# Stage 2: customer selection

Run after the Stage 1 notebook. Inputs: its `stage1_predictions.csv` and the
supplied workbook. Outputs: `stage2_comparison.csv` and `stage2_selection.json`.
The exported main run is QAOA with p=3.


## 1. Imports and settings


In [1]:
from pathlib import Path
from hashlib import sha256
from importlib.metadata import version
from itertools import combinations
from time import perf_counter
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from qiskit import transpile
from qiskit.circuit.library import qaoa_ansatz
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize


In [2]:
WORKBOOK = Path("./bank_campaign_participant_release.xlsx")
STAGE1_CSV = Path("stage1_predictions.csv")
COMPARISON_CSV = Path("stage2_comparison.csv")
RESULTS_JSON = Path("stage2_selection.json")
SEED = 42
MAIN_DEPTH = 3
STARTS = 3
OPTIMIZER_OPTIONS = {"maxiter": 1000, "rhobeg": 0.3, "tol": 1e-5}
BASIS_GATES = ["rz", "sx", "x", "cx"]
sampler = StatevectorSampler(seed=SEED)
np.random.seed(SEED)

print("Python", sys.version.split()[0])
print("Optimizer evaluations per start:", OPTIMIZER_OPTIONS["maxiter"])
print("Sampler default shots:", sampler.default_shots)
packages = ["numpy", "pandas", "scipy", "openpyxl", "qiskit"]
display(pd.DataFrame({"package": packages, "version": [version(p) for p in packages]}))


Python 3.13.2
Optimizer evaluations per start: 1000
Sampler default shots: 1024


,package,version
0,numpy,2.2.6
1,pandas,2.2.3
2,scipy,1.15.2
3,openpyxl,3.1.5
4,qiskit,2.3.1


## 2. Read and prepare the candidates

Sources: `pandas.read_csv`, `read_excel`, and `DataFrame.merge`.
Keep the Stage 1 rank and probabilities. Candidate row i maps to qubit i.


In [3]:
stage1 = pd.read_csv(STAGE1_CSV, float_precision="round_trip")
tables = pd.read_excel(WORKBOOK, sheet_name=["Stage2_Data", "Stage2_Constraints"])
rules = tables["Stage2_Constraints"].set_index("symbol")["value"]
candidate_count = int(rules["N"])
k, budget, capacity_limit = int(rules["K"]), int(rules["C"]), int(rules["H"])
top_ten = stage1.sort_values("rank").head(candidate_count)
candidates = top_ten.merge(tables["Stage2_Data"], on="customer_id", validate="one_to_one")
candidates["expected_net_value"] = (
    candidates["predicted_probability"] * candidates["expected_deposit_value"]
    - candidates["contact_cost"]
)
display(candidates[[
    "rank", "customer_id", "predicted_probability", "expected_deposit_value",
    "contact_cost", "capacity_units", "expected_net_value",
]])


,rank,customer_id,predicted_probability,expected_deposit_value,contact_cost,capacity_units,expected_net_value
0,1,C0081,0.465190,17,4,1,3.908227
1,2,C0031,0.450973,13,2,1,3.862646
2,3,C0079,0.298057,24,3,3,4.153368
3,4,C0170,0.243425,22,1,1,4.355341
4,5,C0327,0.241362,25,1,4,5.034041
5,6,C0193,0.194855,16,3,1,0.117675
6,7,C0256,0.181358,22,1,4,2.989883
7,8,C0349,0.178397,17,1,3,2.032749
8,9,C0183,0.168263,14,2,3,0.355685
9,10,C0378,0.166565,14,3,1,-0.668093


In [4]:
n = len(candidates)
ids = candidates["customer_id"].to_numpy()
values = candidates["expected_net_value"].to_numpy()
costs = candidates["contact_cost"].to_numpy()
capacities = candidates["capacity_units"].to_numpy()
display(pd.Series({
    "candidates": n, "select_exactly": k,
    "budget_limit": budget, "capacity_limit": capacity_limit,
}).to_frame("value"))


,value
candidates,10
select_exactly,6
budget_limit,25
capacity_limit,22


## 3. Exact classical benchmark

Source: `itertools.combinations`. Evaluate every six-customer subset.


In [5]:
def describe_selection(bitstring):
    # Qiskit displays the highest qubit first, so reverse to get candidate order.
    bits = np.array([int(bit) for bit in bitstring[::-1]])
    total_cost = int(bits @ costs)
    total_capacity = int(bits @ capacities)
    return {
        "qiskit_bitstring": bitstring,
        "candidate_order_bits": bitstring[::-1],
        "selected_ids": ids[bits.astype(bool)].tolist(),
        "selected_count": int(bits.sum()),
        "contact_cost_total": total_cost,
        "capacity_units_total": total_capacity,
        "expected_net_value": float(bits @ values),
        "feasible": bool(bits.sum() == k and total_cost <= budget
                         and total_capacity <= capacity_limit),
    }


In [6]:
started = perf_counter()
subsets = []
for chosen in combinations(range(n), k):
    bits = np.zeros(n, dtype=int)
    bits[list(chosen)] = 1
    bitstring = "".join(map(str, bits[::-1]))
    subsets.append(describe_selection(bitstring))
feasible_subsets = [row for row in subsets if row["feasible"]]
classical = sorted(
    feasible_subsets, key=lambda row: (-row["expected_net_value"], row["qiskit_bitstring"]),
)[0]
classical_seconds = perf_counter() - started
display(pd.Series({
    "subsets_examined": len(subsets), "feasible_subsets": len(feasible_subsets),
    "best_value": classical["expected_net_value"],
    "contact_cost": classical["contact_cost_total"],
    "capacity_units": classical["capacity_units_total"],
    "selected_ids": ", ".join(classical["selected_ids"]),
}).to_frame("value"))


,value
subsets_examined,210
feasible_subsets,210
best_value,24.303506
contact_cost,12
capacity_units,14
selected_ids,"C0081, C0031, C0079, C0170, C0327, C0256"


In [7]:
comparison_rows = {}
quality_columns = [
    "method", "best_value", "gap", "p_feasible", "p_optimum",
    "feasible_shots", "optimum_shots",
]


def save_comparison(method, row):
    comparison_rows[method] = row
    table = pd.DataFrame.from_dict(comparison_rows, orient="index").fillna("")
    table = table.rename_axis("method").reset_index()
    table.to_csv(COMPARISON_CSV, index=False)
    display(table.reindex(columns=quality_columns).fillna("").round(4))


save_comparison("exact_classical", {
    "best_value": classical["expected_net_value"], "gap": 0.0,
    "contact_cost": classical["contact_cost_total"],
    "capacity_units": classical["capacity_units_total"],
    "elapsed_seconds": classical_seconds,
})


,method,best_value,gap,p_feasible,p_optimum,feasible_shots,optimum_shots
0,exact_classical,24.3035,0.0,,,,


## 4. Quantum formulation

Let $u_i=v_i/\max_j|v_j|$, where $v_i$ is expected net value. Minimize

$$E(x)=-\sum_i u_i x_i+1.1\left(\sum_i x_i-k\right)^2.$$

The cardinality penalty follows the approach in the
[Qiskit portfolio tutorial](https://qiskit-community.github.io/qiskit-finance/tutorials/01_portfolio_optimization.html).
Since $|u_i|\leq1$, fixing r wrong selections changes the linear term by at
most r but removes penalty $1.1r^2>r$. Thus an infeasible-cardinality state
cannot be the global minimum.

The largest cost and capacity totals for six candidates are below both
limits in this instance. We need only the exactly-six penalty and ten
decision qubits; decoding still checks the original resource limits.


In [8]:
display(pd.DataFrame({
    "resource": ["contact cost", "capacity units"],
    "largest_total_for_six": [int(np.sort(costs)[-k:].sum()), int(np.sort(capacities)[-k:].sum())],
    "limit": [budget, capacity_limit],
}))

value_scale = float(np.max(np.abs(values)))
u = values / value_scale
penalty = 1.1
shift = n / 2 - k
terms = [("Z", [i], float(u[i] / 2 - penalty * shift)) for i in range(n)]
terms += [("ZZ", [i, j], penalty / 2) for i, j in combinations(range(n), 2)]
cost_operator = SparsePauliOp.from_sparse_list(terms, num_qubits=n)
constant_offset = float(-u.sum() / 2 + penalty * (shift**2 + n / 4))


,resource,largest_total_for_six,limit
0,contact cost,17,25
1,capacity units,18,22


Source: `qiskit.quantum_info.SparsePauliOp`. Substitute $x_i=(1-Z_i)/2$.
The identity offset is omitted from circuit evolution and included in the
expectation values below. The binary objective has the same minimizer as
the resulting Ising operator.


In [9]:
indices = np.arange(2**n)
basis_bits = (indices[:, None] >> np.arange(n)) & 1
number_selected = basis_bits.sum(axis=1)
net_value = basis_bits @ values
feasible = (
    (number_selected == k) & (basis_bits @ costs <= budget)
    & (basis_bits @ capacities <= capacity_limit)
)
energy = -(basis_bits @ u) + penalty * (number_selected - k)**2
optimal = feasible & np.isclose(net_value, classical["expected_net_value"], rtol=0, atol=1e-10)
display(pd.Series({
    "decision_qubits": n, "auxiliary_qubits": 0,
    "Z_terms": n, "ZZ_terms": n * (n - 1) // 2,
    "value_scale": value_scale, "penalty": penalty,
}).to_frame("value"))


,value
decision_qubits,10.000000
auxiliary_qubits,0.000000
Z_terms,10.000000
ZZ_terms,45.000000
value_scale,5.034041
penalty,1.100000


## 5. QAOA: p=1, p=2, and p=3

[qiskit.circuit.library.qaoa_ansatz](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.qaoa_ansatz)
supplies the circuit: a Hadamard initial state followed by cost and X-mixer
layers. `Statevector` supplies exact expectations during optimization.
[SciPy COBYLA](https://docs.scipy.org/doc/scipy-1.17.0/reference/optimize.minimize-cobyla.html)
gets at most 1,000 function evaluations per start at every depth.

Use three starts, seed 42: beta in [-pi/4,0], gamma in [0,0.3]. At higher
depths, the first start adds a zero layer to the preceding optimized circuit.
These are initialization ranges, not optimization bounds. Select the lowest
expected penalized energy found; record convergence for every start.


In [10]:
def initial_points(p, previous_angles=None):
    rng = np.random.default_rng(SEED)
    points = [
        np.r_[rng.uniform(-np.pi / 4, 0, p), rng.uniform(0, 0.3, p)]
        for _ in range(STARTS)
    ]
    if previous_angles is not None:
        old_p = len(previous_angles) // 2
        points[0] = np.r_[previous_angles[:old_p], 0, previous_angles[old_p:], 0]
    return points


In [11]:
def optimize_qaoa(p, previous_angles=None):
    circuit = qaoa_ansatz(cost_operator, reps=p)

    def expectation(theta):
        state = Statevector.from_instruction(circuit.assign_parameters(theta))
        return float(state.probabilities() @ energy)

    results, start_reports = [], []
    for start, theta in enumerate(initial_points(p, previous_angles), 1):
        result = minimize(expectation, theta, method="COBYLA", options=OPTIMIZER_OPTIONS)
        results.append(result)
        start_reports.append({
            "start": start, "initial_parameters": theta.tolist(),
            "optimized_parameters": result.x.tolist(), "energy": float(result.fun),
            "evaluations": int(result.nfev), "converged": bool(result.success),
            "message": str(result.message),
        })
        print(f"p={p}, start={start}: {result.nfev} evaluations, converged={result.success}")
    best = min(results, key=lambda result: result.fun)
    return circuit, best, start_reports


[qiskit.primitives.StatevectorSampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler)
measures the optimized circuit. Keep its default 1,024 shots: no shot count
is passed to the constructor or `run`. Keep all counts and select the best
feasible measured state. Displayed bitstrings are q9...q0.


In [12]:
def sample_and_score(bound, probability):
    measured = bound.copy()
    measured.measure_all()
    result = sampler.run([measured]).result()[0].data.meas
    counts = result.get_counts()
    samples = pd.DataFrame([
        {**describe_selection(bitstring), "count": int(count)}
        for bitstring, count in counts.items()
    ])
    valid = samples[samples["feasible"]].sort_values(
        ["expected_net_value", "qiskit_bitstring"], ascending=[False, True],
    )
    selected = describe_selection(valid.iloc[0]["qiskit_bitstring"])
    gap = classical["expected_net_value"] - selected["expected_net_value"]
    statistics = {
        "best_value": selected["expected_net_value"],
        "gap": 0.0 if abs(gap) < 1e-10 else float(gap),
        "p_feasible": float(probability[feasible].sum()),
        "p_optimum": float(probability[optimal].sum()),
        "feasible_shots": int(valid["count"].sum()),
        "optimum_shots": int(sum(count for bits, count in counts.items() if optimal[int(bits, 2)])),
        "shots": int(result.num_shots),
    }
    return selected, statistics, counts


In [13]:
def circuit_resources(circuit):
    return {
        "qubits": circuit.num_qubits, "depth": circuit.depth(),
        "one_qubit_gates": sum(g.operation.num_qubits == 1 for g in circuit.data),
        "two_qubit_gates": sum(g.operation.num_qubits == 2 for g in circuit.data),
        "swaps": int(circuit.count_ops().get("swap", 0)),
        "gate_counts": dict(circuit.count_ops()),
    }


In [14]:
runs = {}


def record_qaoa(p, circuit, result, start_reports, started):
    bound = circuit.assign_parameters(result.x)
    probability = Statevector.from_instruction(bound).probabilities()
    selected, statistics, counts = sample_and_score(bound, probability)
    compiled = transpile(
        bound, basis_gates=BASIS_GATES, optimization_level=1, seed_transpiler=SEED,
    )
    resources = {"original": circuit_resources(bound), "transpiled": circuit_resources(compiled)}
    row = {
        **statistics, "contact_cost": selected["contact_cost_total"],
        "capacity_units": selected["capacity_units_total"],
        "expected_energy": float(result.fun),
        "mean_feasible_value": float(probability[feasible] @ net_value[feasible] / statistics["p_feasible"]),
        "evaluations": sum(start["evaluations"] for start in start_reports),
        "winner_converged": bool(result.success), "parameters": circuit.num_parameters,
        "qubits": n, "original_depth": resources["original"]["depth"],
        "transpiled_depth": resources["transpiled"]["depth"],
        "cx_gates": resources["transpiled"]["gate_counts"].get("cx", 0),
        "elapsed_seconds": perf_counter() - started,
    }
    runs[p] = {
        "p": p, **selected, "statistics": row,
        "parameter_order": [str(parameter) for parameter in circuit.parameters],
        "optimized_parameters": result.x.tolist(), "starts": start_reports,
        "measurement_counts": {str(bits): int(count) for bits, count in counts.items()},
        "circuit_resources": resources,
    }
    save_comparison(f"qaoa_p{p}", row)


In [15]:
started = perf_counter()
circuit1, result1, starts1 = optimize_qaoa(1)
record_qaoa(1, circuit1, result1, starts1, started)


p=1, start=1: 47 evaluations, converged=True
p=1, start=2: 45 evaluations, converged=True
p=1, start=3: 42 evaluations, converged=True


,method,best_value,gap,p_feasible,p_optimum,feasible_shots,optimum_shots
0,exact_classical,24.3035,0.0,,,,
1,qaoa_p1,24.3035,0.0,0.53741,0.003181,544.0,5.0


In [16]:
started = perf_counter()
circuit2, result2, starts2 = optimize_qaoa(2, result1.x)
record_qaoa(2, circuit2, result2, starts2, started)


p=2, start=1: 1000 evaluations, converged=False
p=2, start=2: 404 evaluations, converged=True
p=2, start=3: 563 evaluations, converged=True


,method,best_value,gap,p_feasible,p_optimum,feasible_shots,optimum_shots
0,exact_classical,24.3035,0.0,,,,
1,qaoa_p1,24.3035,0.0,0.53741,0.003181,544.0,5.0
2,qaoa_p2,24.3035,0.0,0.608252,0.004549,641.0,2.0


In [17]:
started = perf_counter()
circuit3, result3, starts3 = optimize_qaoa(3, result2.x)
record_qaoa(3, circuit3, result3, starts3, started)


p=3, start=1: 1000 evaluations, converged=False
p=3, start=2: 732 evaluations, converged=True
p=3, start=3: 1000 evaluations, converged=False


,method,best_value,gap,p_feasible,p_optimum,feasible_shots,optimum_shots
0,exact_classical,24.3035,0.0,,,,
1,qaoa_p1,24.3035,0.0,0.53741,0.003181,544.0,5.0
2,qaoa_p2,24.3035,0.0,0.608252,0.004549,641.0,2.0
3,qaoa_p3,24.3035,0.0,0.618569,0.0085,642.0,9.0


Transpilation of the bound optimized circuits: `qiskit.transpile`, basis rz/sx/x/cx, optimization level 1,
seed 42, no hardware target. Counts exclude measurements. All execution is
ideal simulation. Sampling fields in the classical row are left blank.


In [18]:
resources_columns = [
    "method", "evaluations", "winner_converged", "qubits", "parameters",
    "original_depth", "transpiled_depth", "cx_gates", "elapsed_seconds",
]
comparison = pd.DataFrame.from_dict(comparison_rows, orient="index").fillna("")
comparison = comparison.rename_axis("method").reset_index()
display(comparison.reindex(columns=resources_columns).round(4))


,method,evaluations,winner_converged,qubits,parameters,original_depth,transpiled_depth,cx_gates,elapsed_seconds
0,exact_classical,,,,,,,,0.0054
1,qaoa_p1,134.0,True,10.0,2.0,20.0,59.0,90.0,1.7104
2,qaoa_p2,1967.0,True,10.0,4.0,32.0,94.0,180.0,14.6356
3,qaoa_p3,2732.0,True,10.0,6.0,44.0,129.0,270.0,29.1827


## 6. Main selection and export

Export p=3 as agreed. Its selected bitstring must be one actually measured;
use its original, unnormalized expected net value. The JSON keeps all three
runs, their parameters and counts, and the exact classical reference.


In [19]:
main_run = runs[MAIN_DEPTH]
selected_customers = candidates[candidates["customer_id"].isin(main_run["selected_ids"])]
display(selected_customers[[
    "customer_id", "predicted_probability", "expected_deposit_value",
    "contact_cost", "capacity_units", "expected_net_value",
]])
display(pd.Series({
    "main_depth": MAIN_DEPTH, "selected_ids": ", ".join(main_run["selected_ids"]),
    "net_value": main_run["expected_net_value"],
    "contact_cost": main_run["contact_cost_total"],
    "capacity_units": main_run["capacity_units_total"],
    "gap_to_classical": main_run["statistics"]["gap"],
}).to_frame("value"))


,customer_id,predicted_probability,expected_deposit_value,contact_cost,capacity_units,expected_net_value
0,C0081,0.465190,17,4,1,3.908227
1,C0031,0.450973,13,2,1,3.862646
2,C0079,0.298057,24,3,3,4.153368
3,C0170,0.243425,22,1,1,4.355341
4,C0327,0.241362,25,1,4,5.034041
6,C0256,0.181358,22,1,4,2.989883


,value
main_depth,3
selected_ids,"C0081, C0031, C0079, C0170, C0327, C0256"
net_value,24.303506
contact_cost,12
capacity_units,14
gap_to_classical,0.0


In [20]:
formulation = {
    "objective": "-sum(u_i*x_i) + penalty*(sum(x_i)-k)^2",
    "value_scale": value_scale, "penalty": penalty, "select_exactly": k,
    "budget_limit": budget, "capacity_limit": capacity_limit,
    "budget_capacity_handling": "redundant for this six-customer pool; checked in decoding",
    "auxiliary_qubits": 0, "ising_constant_offset": constant_offset,
    "cost_operator_without_constant": [
        [label, float(np.real(coefficient))] for label, coefficient in cost_operator.to_list()
    ],
}
setup = {
    "main_depth": MAIN_DEPTH, "depths": [1, 2, 3],
    "optimizer": "scipy.optimize.minimize COBYLA", "optimizer_options": OPTIMIZER_OPTIONS,
    "starts_per_depth": STARTS, "initial_state": "Hadamards", "mixer": "global X rotations",
    "initial_beta_range": [-float(np.pi / 4), 0], "initial_gamma_range": [0, 0.3],
    "deeper_first_start": "previous optimized angles with a zero additional layer",
    "sampling": "StatevectorSampler default; no shot-count override",
    "shots": main_run["statistics"]["shots"],
    "seeds": {"initialization": SEED, "sampler": SEED, "transpiler": SEED},
    "backend": "Qiskit exact statevector simulation on CPU", "device_noise_model": None,
    "physical_hardware_used": False, "basis_gates": BASIS_GATES, "optimization_level": 1,
    "hardware_target": None, "bit_order": "candidate i maps to qubit i; display q9...q0",
    "stage1_predictions_sha256": sha256(STAGE1_CSV.read_bytes()).hexdigest(),
    "workbook_sha256": sha256(WORKBOOK.read_bytes()).hexdigest(),
}


In [21]:
report = {
    "setup": setup, "formulation": formulation, "main_depth": MAIN_DEPTH,
    "selected_ids": main_run["selected_ids"],
    "expected_net_value": main_run["expected_net_value"],
    "contact_cost_total": main_run["contact_cost_total"],
    "capacity_units_total": main_run["capacity_units_total"],
    "qiskit_bitstring": main_run["qiskit_bitstring"],
    "absolute_gap_to_classical_optimum": main_run["statistics"]["gap"],
    "candidates_in_qubit_order": candidates.to_dict(orient="records"),
    "classical_reference": {**classical, "subsets_examined": len(subsets),
                            "feasible_subsets": len(feasible_subsets)},
    "runs": list(runs.values()),
    "comparison": [{"method": method, **row} for method, row in comparison_rows.items()],
    "versions": {package: version(package) for package in packages},
}
RESULTS_JSON.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
print(f"Saved {len(comparison_rows)} comparisons to {COMPARISON_CSV}")
print(f"Saved main p={MAIN_DEPTH} selection and all runs to {RESULTS_JSON}")


Saved 4 comparisons to stage2_comparison.csv
Saved main p=3 selection and all runs to stage2_selection.json
